# 14d · Un aggiornamento difettoso: il device lo prova e torna indietro

Il notebook 14c ha installato la **2.0.0**. Ora il job consegna la **3.0.0**, che è firmata bene ma **difettosa** (lancia un errore). Lo stesso device:
1. chiede, e **verifica la firma** (la firma è valida: chi l'ha rilasciata è chi dice);
2. scarica e controlla l'hash (il file è quello del manifest);
3. **tiene da parte la 2.0.0**, installa la 3.0.0 e la **prova**: fallisce;
4. **ripristina** la 2.0.0 e **riporta `FAILED`** con il codice di errore.

Il servizio mostra il fallimento **per device**. **Non c'è rollback automatico del job**: il ritorno indietro lo fa il **device**, perché la firma dice *chi* ha rilasciato il file, non che *funzioni*.

**Serve:** il notebook 14c eseguito fino al passo 8 (il device alla 2.0.0 e la software revision a 2.0.0 in ADR).
**Da dire.** "Il job non sa se l'aggiornamento funziona. Lo sa il device, che prova e torna indietro: è lui che decide."


In [ ]:
# Carica la libreria e legge lo scenario (notebooks\_scenario.psd1): da qui in poi si usa $cfg
. ([scriptblock]::Create([IO.File]::ReadAllText((Get-ChildItem . , .\notebooks , .. -Filter _start.ps1 -ErrorAction SilentlyContinue | Select-Object -First 1).FullName)))

## 0 · Il device e i nomi


In [ ]:
$id = 'device-da-aggiornare'                 # il device dell'aggiornamento (Contoso, RobotV1, software revision 1.0.0)
$provider = 'Contoso'; $nome = 'termostato'    # l'identita' dell'aggiornamento: provider / nome / versione
$ns = $adrName
$cartella = (Get-ChildItem (Join-Path $out "devices\$id") -Directory | Sort-Object Name | Select-Object -Last 1).FullName     # i certificati del device (notebook 02c)
$stato = Join-Path $out "su-agent\$id"                                       # dove il device tiene cio' che ha installato
$agente = (Resolve-Path (Join-Path $base '..\device\su-agent\su-agent.cs')).Path
$idScope = az iot dps show --name $dpsName --resource-group $rg --query properties.idScope -o tsv --only-show-errors
if (-not (Test-Path (Join-Path $cartella 'bootstrap-chain.pem'))) { throw "Il device $id non ha il certificato di bootstrap: esegui il notebook 02c." }
"device: $id | certificati: $cartella"
"id scope DPS: $idScope"


## 1 · Il punto di partenza
Il device ha la 2.0.0 installata e in ADR la sua software revision è 2.0.0. Il gruppo nuovo prende i device **a 2.0.0**: il vecchio gruppo (a 1.0.0) non vale più.


In [ ]:
$gruppo = 'gruppo-robotv1-rev2'; $job = 'aggiorna-termostato-3'
"installato sul device: $(Get-Content (Join-Path $stato 'installed.json') -Raw)"
"payload attuale: " + (pwsh -NoProfile -Command ". '$(Join-Path $stato 'payload\current.ps1')'; (Get-Telemetry).payloadVersion")
az iot adr ns device show --name $id --namespace $ns --resource-group $rg --query "{device:name, softwareRevision:properties.softwareRevision}" -o table --only-show-errors
Remove-AdrGroup -Name $gruppo -Namespace $ns -ResourceGroup $rg
$errore = New-AdrGroup -Name $gruppo -Query "WHERE manufacturer = 'Contoso' AND model = 'RobotV1' AND softwareRevision = '2.0.0'" -Namespace $ns -ResourceGroup $rg
if ($errore) { throw "Gruppo rifiutato: $errore" }
"Membri di ${gruppo}: $((Get-AdrGroupMembers -Name $gruppo -Namespace $ns -ResourceGroup $rg) -join ', ')"


## 2 · Job e run per la 3.0.0


In [ ]:
az iot adr ns job delete --name $job --ns $ns --resource-group $rg --yes -o none --only-show-errors 2>$null
az iot adr ns job create --name $job --ns $ns --resource-group $rg --type SoftwareUpdate --target-group-name $gruppo `
    --update-id-provider $provider --update-id-name $nome --update-id-version 3.0.0 --description "Termostato 3.0.0 (difettoso)" -o none --only-show-errors
Assert-Az "Creazione del job"
az iot adr ns job schedule --name $job --ns $ns --resource-group $rg --run-name run-1 -o none --only-show-errors
Assert-Az "Programmazione della run"
for ($i = 0; $i -lt 12; $i++) {
    Start-Sleep -Seconds 5
    $r = az iot adr ns job run show --name run-1 --job-name $job --ns $ns --resource-group $rg --query "{stato:properties.status, errore:properties.error.code}" -o json --only-show-errors | ConvertFrom-Json
    if ($r.stato -ne 'Scheduled') { break }
}
"run-1: $($r.stato) $($r.errore)"


## 3 · Il device chiede, prova, torna indietro
Guarda le righe: firma e hash vanno bene, poi **il test fallisce** e il device ripristina la versione buona.


In [ ]:
Invoke-DeviceUpdateCheck -Device $id -Folder $cartella -State $stato -IdScope $idScope -AgentFile $agente


## 4 · Cosa dice il servizio, e cosa c'è sul device
Per il device: `Failed`, origine `DEVICE`, codice `-5` e codice esteso `50000000` (la **faccia 5 è l'installazione**, nei 4 bit alti). Il device resta alla **2.0.0**: la versione buona è stata ripristinata. In ADR la software revision è ancora 2.0.0, e il job **non** ha riprovato.


In [ ]:
Start-Sleep -Seconds 30      # il riepilogo si aggiorna qualche decina di secondi DOPO il risultato per device
az iot adr ns job run summary --name run-1 --job-name $job --ns $ns --resource-group $rg -o json --only-show-errors
az iot adr ns job run results --name run-1 --job-name $job --ns $ns --resource-group $rg -o json --only-show-errors | ConvertFrom-Json |
    ForEach-Object { [pscustomobject]@{ device = ($_.resourceId -split '/')[-1]; stato = $_.status; origine = $_.installResult.failureOrigin; codice = $_.installResult.resultCode; esteso = $_.installResult.extendedResultCodes } } | Format-Table | Out-String
"installato sul device: $(Get-Content (Join-Path $stato 'installed.json') -Raw)"
"payload attuale: " + (pwsh -NoProfile -Command ". '$(Join-Path $stato 'payload\current.ps1')'; (Get-Telemetry).payloadVersion")


## Pulizia
Si chiude la run, si cancellano job, run e gruppi, e il device torna alla software revision `1.0.0` e dimentica quello che aveva installato, così i notebook si rilanciano.


In [ ]:
foreach ($j in 'aggiorna-termostato-2', 'aggiorna-termostato-3') {
    foreach ($r in @(az iot adr ns job run list --job-name $j --ns $ns --resource-group $rg --query "[].name" -o tsv --only-show-errors 2>$null)) {
        az iot adr ns job run cancel --name $r --job-name $j --ns $ns --resource-group $rg --yes -o none --only-show-errors 2>$null
        az iot adr ns job run delete --name $r --job-name $j --ns $ns --resource-group $rg --yes -o none --only-show-errors 2>$null
    }
    az iot adr ns job delete --name $j --ns $ns --resource-group $rg --yes -o none --only-show-errors 2>$null
}
foreach ($g in 'gruppo-robotv1-rev1', 'gruppo-robotv1-rev2') { Remove-AdrGroup -Name $g -Namespace $ns -ResourceGroup $rg }
Set-AdrDeviceProperties -Name $id -Namespace $ns -ResourceGroup $rg -SoftwareRevision '1.0.0'
if (Test-Path $stato) { Remove-Item $stato -Recurse -Force }
"Pulito: job, run e gruppi cancellati; $id alla software revision 1.0.0."


## Cosa abbiamo visto
- La **firma** valida non garantisce che l'aggiornamento **funzioni**: il device lo **prova**, e se fallisce **ripristina** la versione buona.
- Il device **riporta** il fallimento (origine `DEVICE`, codice negativo, codice esteso con la faccia): il servizio lo mostra **per device**.
- **Il job non riprova e non torna indietro**: per riprovare serve un nuovo job. Il rollback è del device.
- Un solo device ha chiesto: gli altri del gruppo (se ce ne fossero) restano *non applicati* finché non chiedono.

**Prossimo notebook: 16, il file firmato in locale (la stessa idea, senza Azure).**
